# Metabo — Metabolic Wellness Index Engine
## Google Colab Notebook

**What this does:**
1. Generates 30 days of simulated wearable data (HRV, sleep, steps, resting heart rate)
2. Builds your personal baseline from the first 7 days
3. Calculates your Metabolic Wellness Index (Tier 1 / 2 / 3)
4. Updates Google Sheets so your FlutterFlow app can read it

**How to use:** Click Runtime → Run all. That's it.

---

## STEP 1: Install required libraries

In [ ]:
!pip install gspread pandas numpy Faker --quiet

## STEP 2: Enter your profile data

In [ ]:
# EDIT THESE VALUES for your profile

age = 35             # your age (25–45)
sex = "male"        # "male" or "female"
ethnicity = "chinese"  # "chinese" / "malay" / "indian" / "others"
height_cm = 170      # height in cm
weight_kg = 75      # weight in kg
bmi = weight_kg / ((height_cm/100) ** 2)
family_history_t2d = True   # True if a parent or sibling has Type 2 diabetes
gestational_diabetes = False  # True if you have had gestational diabetes (women only)

print(f"BMI: {bmi:.1f}")
print(f"Profile saved: Age {age}, {sex}, {ethnicity}, BMI {bmi:.1f}, Family Hx: {family_history_t2d}")

## STEP 3: Generate 30 days of simulated wearable data

In [ ]:
import random
import numpy as np
from faker import Faker
fake = Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)

# Generate 30 days of daily data
days = []
for day in range(1, 31):
    # HRV (RMSSD in ms) — higher is better
    # Healthy adult range: 20–80ms, baseline ~55ms
    hrv = random.gauss(55, 12)
    hrv = max(20, min(100, hrv))  # clamp to realistic range
    
    # Resting heart rate (bpm) — lower is better
    # Healthy adult range: 50–75bpm, baseline ~62
    rhr = random.gauss(62, 5)
    rhr = max(48, min(85, rhr))
    
    # Sleep efficiency (%) — higher is better
    # Healthy range: 75–95%, baseline ~85%
    sleep_eff = random.gauss(85, 6)
    sleep_eff = max(70, min(97, sleep_eff))
    
    # Total steps — baseline ~8,000
    steps = int(random.gauss(8000, 2000))
    steps = max(2000, min(20000, steps))
    
    # Sleep quality rating (1–5 self-report)
    sleep_quality = round(random.gauss(3.5, 0.7))
    sleep_quality = max(1, min(5, sleep_quality))
    
    # Energy level (1–5)
    energy = round(random.gauss(3.2, 0.8))
    energy = max(1, min(5, energy))
    
    # Stress level (1–5)
    stress = round(random.gauss(3.0, 0.9))
    stress = max(1, min(5, stress))
    
    # Hawker meals eaten today (0–5)
    hawker_meals = random.randint(0, 5)
    
    days.append({
        "day": day,
        "hrv_rmssd": round(hrv, 1),
        "resting_hr": round(rhr, 1),
        "sleep_efficiency": round(sleep_eff, 1),
        "steps": steps,
        "sleep_quality": int(sleep_quality),
        "energy": int(energy),
        "stress": int(stress),
        "hawker_meals": hawker_meals
    })

print(f"Generated {len(days)} days of wearable data")
print(f"Sample Day 1: HRV={days[0]['hrv_rmssd']}ms, RHR={days[0]['resting_hr']}bpm, Steps={days[0]['steps']}, Sleep Eff={days[0]['sleep_efficiency']}%")

## STEP 4: Calculate your personal baseline (Days 1–7)

In [ ]:
# First 7 days establish the personal baseline
baseline_days = days[:7]

baseline_hrv = np.mean([d["hrv_rmssd"] for d in baseline_days])
baseline_rhr = np.mean([d["resting_hr"] for d in baseline_days])
baseline_sleep_eff = np.mean([d["sleep_efficiency"] for d in baseline_days])
baseline_steps = np.mean([d["steps"] for d in baseline_days])
baseline_sleep_quality = np.mean([d["sleep_quality"] for d in baseline_days])
baseline_energy = np.mean([d["energy"] for d in baseline_days])
baseline_stress = np.mean([d["stress"] for d in baseline_days])
baseline_hawker = np.mean([d["hawker_meals"] for d in baseline_days])

print("=== YOUR PERSONAL BASELINE (Days 1–7) ===")
print(f"HRV (RMSSD):       {baseline_hrv:.1f} ms")
print(f"Resting HR:        {baseline_rhr:.1f} bpm")
print(f"Sleep Efficiency:   {baseline_sleep_eff:.1f}%")
print(f"Steps:             {baseline_steps:.0f} per day")
print(f"Sleep Quality:     {baseline_sleep_quality:.1f} / 5")
print(f"Energy:            {baseline_energy:.1f} / 5")
print(f"Stress:            {baseline_stress:.1f} / 5")
print(f"Hawker Meals:      {baseline_hawker:.1f} per day")

## STEP 5: Calculate today's deviation scores (Days 8–30)

In [ ]:
# Calculate how far today's metrics deviate from personal baseline
# Positive = improved, Negative = worse than baseline

for day_data in days[7:]:
    day = day_data["day"]
    
    # HRV deviation — lower HRV = metabolic stress signal
    hrv_dev = (day_data["hrv_rmssd"] - baseline_hrv) / baseline_hrv * 100
    
    # RHR deviation — higher RHR = recovery deficit
    rhr_dev = (day_data["resting_hr"] - baseline_rhr) / baseline_rhr * 100
    
    # Sleep efficiency deviation
    sleep_dev = (day_data["sleep_efficiency"] - baseline_sleep_eff) / baseline_sleep_eff * 100
    
    # Steps deviation
    steps_dev = (day_data["steps"] - baseline_steps) / baseline_steps * 100
    
    # Hawker excess (over 2 meals/day baseline)
    hawker_excess = max(0, day_data["hawker_meals"] - 2)
    
    day_data["hrv_dev_pct"] = round(hrv_dev, 1)
    day_data["rhr_dev_pct"] = round(rhr_dev, 1)
    day_data["sleep_dev_pct"] = round(sleep_dev, 1)
    day_data["steps_dev_pct"] = round(steps_dev, 1)
    day_data["hawker_excess"] = hawker_excess

print("Deviation scores calculated for Days 8–30")
print(f"Day 8: HRV {days[7]['hrv_rmssd']}ms ({days[7]['hrv_dev_pct']:+.1f}% vs baseline)")
print(f"Day 15: HRV {days[14]['hrv_rmssd']}ms ({days[14]['hrv_dev_pct']:+.1f}% vs baseline)")
print(f"Day 22: HRV {days[21]['hrv_rmssd']}ms ({days[21]['hrv_dev_pct']:+.1f}% vs baseline)")
print(f"Day 30: HRV {days[29]['hrv_rmssd']}ms ({days[29]['hrv_dev_pct']:+.1f}% vs baseline)")

## STEP 6: Calculate your Layer 1 — Demographic Risk Score

This estimates your underlying metabolic risk based on factors that don't change daily.
Based on NHANES Cox PH model simplified for prototype.

| Risk Factor | Effect |
|------------|--------|
| Age 35–40 | +1 point |
| BMI ≥ 25 (Asian cutoff) | +2 points |
| BMI ≥ 30 | +3 points |
| Indian ethnicity | +2 points |
| Malay ethnicity | +1.5 points |
| Family history T2D | +2 points |
| Gestational diabetes | +2 points |

In [ ]:
# Layer 1: Demographic Risk Score

risk_score = 0

if age >= 35:
    risk_score += 1

if bmi >= 30:
    risk_score += 3
elif bmi >= 25:  # Asian BMI cutoff
    risk_score += 2

if ethnicity == "indian":
    risk_score += 2
elif ethnicity == "malay":
    risk_score += 1.5

if family_history_t2d:
    risk_score += 2

if gestational_diabetes:
    risk_score += 2

# Map risk score to tier (0–10 scale normalized)
# Higher score = higher metabolic risk
if risk_score <= 2:
    tier = "Tier 1 — Wellness"
    tier_color = "🟢"
elif risk_score <= 5:
    tier = "Tier 2 — Monitoring Recommended"
    tier_color = "🟡"
else:
    tier = "Tier 3 — Lifestyle Attention"
    tier_color = "🔴"

print("=== LAYER 1: DEMOGRAPHIC RISK SCORE ===")
print(f"Total risk score: {risk_score}")
print(f"{tier_color} {tier}")

## STEP 7: Calculate Layer 2 — 7-Day Anomaly Score

In [ ]:
# Layer 2: Recent 7-day deviation from personal baseline
# This is the "anomaly detection" — are you trending worse than YOUR normal?

recent_7 = days[-7:]  # last 7 days

avg_hrv_dev = np.mean([d["hrv_dev_pct"] for d in recent_7])
avg_rhr_dev = np.mean([d["rhr_dev_pct"] for d in recent_7])
avg_sleep_dev = np.mean([d["sleep_dev_pct"] for d in recent_7])
avg_steps_dev = np.mean([d["steps_dev_pct"] for d in recent_7])
avg_hawker_excess = np.mean([d["hawker_excess"] for d in recent_7])

# Composite anomaly score: negative = worse than baseline
# HRV and sleep efficiency deviations are most predictive
anomaly_score = (
    (avg_hrv_dev * 0.35) +       # HRV is the primary signal
    (avg_rhr_dev * -0.15) +       # RHR going UP is bad (negative because lower RHR is better)
    (avg_sleep_dev * 0.25) +       # Sleep efficiency
    (avg_steps_dev * 0.10) +       # Steps matter but less than HRV/sleep
    (avg_hawker_excess * -3)        # Each excess hawker meal = -3 points
)

print("=== LAYER 2: 7-DAY ANOMALY SCORE ===")
print(f"Avg HRV deviation (7d):  {avg_hrv_dev:+.1f}%")
print(f"Avg RHR deviation (7d):  {avg_rhr_dev:+.1f}%")
print(f"Avg Sleep deviation (7d):  {avg_sleep_dev:+.1f}%")
print(f"Avg Steps deviation (7d): {avg_steps_dev:+.1f}%")
print(f"Avg Hawker excess (7d):   {avg_hawker_excess:.1f} meals/day over baseline")
print(f"Composite anomaly score:   {anomaly_score:.1f}")

if anomaly_score >= 5:
    trend = "↑ IMPROVING — your metrics are better than your personal baseline"
elif anomaly_score <= -5:
    trend = "↓ NEEDS ATTENTION — your metrics are below your personal baseline"
else:
    trend = "→ STABLE — you are holding steady near your personal baseline"

print(f"Trend: {trend}")

## STEP 8: Your Metabolic Wellness Index

In [ ]:
# Combine Layer 1 (demographic risk) + Layer 2 (recent trend)
# Final MWI is your risk tier adjusted by recent behaviour

# Adjust tier based on recent trend
if anomaly_score <= -8:
    # Significantly worse than baseline — escalate tier
    if "Tier 1" in tier:
        final_tier = "Tier 2 — Monitoring Recommended"
        final_tier_color = "🟡"
        tier_message = "Your recent metrics are below your personal baseline. Consider more rest and movement this week."
    elif "Tier 2" in tier:
        final_tier = "Tier 3 — Lifestyle Attention"
        final_tier_color = "🔴"
        tier_message = "Your recent trends are declining. Consider consulting your GP for a health check."
    else:
        final_tier = "Tier 3 — Lifestyle Attention"
        final_tier_color = "🔴"
        tier_message = "Your metrics need attention. Please consult a healthcare provider."
elif anomaly_score >= 8:
    # Significantly better than baseline — de-escalate
    if "Tier 3" in tier:
        final_tier = "Tier 2 — Monitoring Recommended"
        final_tier_color = "🟡"
        tier_message = "Great progress! Your recent metrics are better than your baseline. Keep it up."
    else:
        final_tier = tier
        final_tier_color = tier_color
        tier_message = "Excellent trends — you're doing better than your usual. Your body is recovering well."
else:
    final_tier = tier
    final_tier_color = tier_color
    tier_message = "Your metrics are stable. Keep maintaining your current habits."

print("=" * 50)
print("   METABOLIC WELLNESS INDEX")
print("=" * 50)
print(f"{final_tier_color} {final_tier}")
print()
print(tier_message)
print()
print("=== YOUR NUMBERS TODAY ===")
today = days[-1]
print(f"HRV today:         {today['hrv_rmssd']} ms ({today['hrv_dev_pct']:+.1f}% vs baseline)")
print(f"Resting HR today:   {today['resting_hr']} bpm ({today['rhr_dev_pct']:+.1f}% vs baseline)")
print(f"Sleep efficiency:   {today['sleep_efficiency']}% ({today['sleep_dev_pct']:+.1f}% vs baseline)")
print(f"Steps today:       {today['steps']} ({today['steps_dev_pct']:+.1f}% vs baseline)")
print(f"Hawker meals:      {today['hawker_meals']} today ({today['hawker_excess']} over healthy baseline)")
print()
print(f"30-day avg HRV:    {np.mean([d['hrv_rmssd'] for d in days]):.1f} ms")
print(f"30-day avg steps:  {np.mean([d['steps'] for d in days]):.0f}")

## STEP 9: Today's personalised recommendation

In [ ]:
# Rule-based recommendation engine (Layer 2 MVP)
# This is what shows up in the morning briefing

recommendations = []

if today['hrv_dev_pct'] < -15:
    recommendations.append("🔴 Your HRV is significantly below your baseline — consider a rest day today.")
elif today['hrv_dev_pct'] < -5:
    recommendations.append("🟡 Your HRV is slightly below usual — a light walk or stretch is better than nothing today.")

if today['steps_dev_pct'] < -20:
    recommendations.append("🚶 You're well below your step baseline — a 20-minute walk is linked to better recovery.")

if today['hawker_excess'] >= 3:
    recommendations.append("🍜 You've had several hawker meals recently — try a lower-GI option like brown rice or more vegetables tomorrow.")
elif today['hawker_excess'] >= 1:
    recommendations.append("🍜 A few hawker meals this week — you're within range, but vary your carbs where you can.")

if today['sleep_dev_pct'] < -10:
    recommendations.append("😴 Your sleep efficiency is lower than usual — consider no screens 30 minutes before bed tonight.")

if "Tier 3" in final_tier and today['hawker_excess'] >= 2:
    recommendations.append("⚕️ Your metrics and risk profile suggest a routine health check — a GP visit can give you a complete picture.")

if not recommendations:
    recommendations.append("✅ You're tracking well today. Keep maintaining your current habits — consistency is the habit.")

print("=== TODAY'S RECOMMENDATION ===")
for r in recommendations:
    print(r)

## STEP 10: Connect to Google Sheets (OPTIONAL)

Follow these steps ONLY if you want the FlutterFlow app to read live data:

1. Go to [sheets.google.com](https://sheets.google.com) and create a new sheet named "Metabo Data"
2. Create 4 tabs named exactly: `Profile`, `DailyData`, `CheckIns`, `MWI`
3. In the `DailyData` tab, add this header row in Row 1: `day, hrv_rmssd, resting_hr, sleep_efficiency, steps, sleep_quality, energy, stress, hawker_meals, hrv_dev_pct, rhr_dev_pct, sleep_dev_pct, steps_dev_pct, hawker_excess`
4. In the `MWI` tab, add this header row in Row 1: `tier, tier_message, anomaly_score, risk_score, final_tier, recommendation`
5. In the `Profile` tab, add headers: `age, sex, ethnicity, bmi, family_history_t2d, gestational_diabetes`
6. Share the sheet with "Anyone with the link can view"
7. Copy your sheet URL and paste it in the code cell below

Only do this step AFTER creating the sheet — skip if you just want to see the notebook output.

In [ ]:
# PASTE YOUR GOOGLE SHEET URL HERE (the one you just created)
SHEET_URL = ""

if SHEET_URL == "":
    print("No sheet URL provided — skipping Sheets upload.")
    print("Your Metabolic Wellness Index is ready above.")
    print("To connect to FlutterFlow, add your Google Sheet URL above and run this cell again.")
else:
    import gspread
    from google.colab import auth
    auth.authenticate_user()
    gc = gspread.authorize(creds)
    
    # Parse sheet ID from URL
    import re
    sheet_id = re.search(r'/d/([^/]+)', SHEET_URL)
    if sheet_id:
        sheet_id = sheet_id.group(1)
        sh = gc.open_by_key(sheet_id)
        
        # Write profile
        profile_sheet = sh.worksheet("Profile")
        profile_sheet.update([[age, sex, ethnicity, round(bmi,1), family_history_t2d, gestational_diabetes]], major_dimension='ROWS')
        
        # Write daily data (all 30 days)
        daily_sheet = sh.worksheet("DailyData")
        daily_rows = []
        for d in days:
            daily_rows.append([
                d["day"], d["hrv_rmssd"], d["resting_hr"], d["sleep_efficiency"],
                d["steps"], d["sleep_quality"], d["energy"], d["stress"],
                d["hawker_meals"], d["hrv_dev_pct"], d["rhr_dev_pct"],
                d["sleep_dev_pct"], d["steps_dev_pct"], d["hawker_excess"]
            ])
        daily_sheet.update(daily_rows, major_dimension='ROWS')
        
        # Write MWI result
        mwi_sheet = sh.worksheet("MWI")
        mwi_sheet.update([[
            tier, tier_message, round(anomaly_score, 1),
            round(risk_score, 1), final_tier, "; ".join(recommendations)
        ]], major_dimension='ROWS')
        
        print("✅ Data uploaded to Google Sheets!")
        print(f"📊 Sheet URL: {SHEET_URL}")
        print("\nNow open FlutterFlow and connect your app to this sheet.")
    else:
        print("Could not parse sheet URL. Make sure you paste the full URL from your browser.")

---

## How to use this notebook

1. **Open Google Colab**: Go to [colab.google](https://colab.google)
2. **Upload this file**: Click File → Upload notebook → select this `.ipynb` file
3. **Customise your profile**: Edit the values in STEP 2 (age, sex, ethnicity, etc.)
4. **Run all**: Click Runtime → Run all
5. **Get your MWI**: Scroll up to see your Metabolic Wellness Index and today's recommendation
6. **Connect to Google Sheets** (optional): Create a Google Sheet with 4 tabs and paste the URL in STEP 10

**To demo the full app experience:**
- Run the notebook once to populate your baseline (Days 1–7)
- Come back tomorrow and run it again — the model will show Day 2 data
- Or change the fake data in STEP 3 to simulate different scenarios

**Scenarios to demo:**
- Set hawker_meals to 5 for 3 consecutive days → Hawker warning appears
- Set HRV to 35ms → Recovery alert appears
- Set ethnicity = 'indian' and family_history_t2d = True → Tier 3